# 07 — Testar tools e alertas

Sanity check de cada ferramenta + lógica determinística. Não usa LLM — só valida que as funções devolvem o esperado.

**Pré-requisitos:**
1. Notebook `05_gerar_dados_mock.ipynb` rodado (`hospital.db` existe).
2. Notebook `06_indexar_protocolos.ipynb` rodado (Chroma indexado) — opcional, só pra testar `buscar_protocolo`.
3. Pasta `lib/` em `/MyDrive/AssistenteHospitalar/lib/` (Opção A do README).

In [ ]:
!pip install -q faker pydantic langchain langchain-community langchain-huggingface chromadb sentence-transformers

In [ ]:
import os, sys
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

DRIVE_BASE = '/content/drive/MyDrive/AssistenteHospitalar'
LIB_PATH   = DRIVE_BASE           # lib/ deve estar em /MyDrive/AssistenteHospitalar/lib/
DB_PATH    = f'{DRIVE_BASE}/files/hospital.db'
CHROMA_DIR = f'{DRIVE_BASE}/files/chroma'
COLLECTION = 'protocolos_saude_mulher'

sys.path.insert(0, LIB_PATH)
os.environ['HOSPITAL_DB_PATH'] = DB_PATH

from lib import db, tools, alertas
conn = db.connect()
print('Conectado a', DB_PATH)

In [ ]:
# 1) consultar_prontuario
import json
print(json.dumps(tools.consultar_prontuario(1, conn), indent=2, ensure_ascii=False))

In [ ]:
# 2) historico_exames — todos e filtrados por tipo
pid = 1
print('Todos:', tools.historico_exames(pid, conn))
print('\nSó papanicolau:', tools.historico_exames(pid, conn, tipo='papanicolau'))

In [ ]:
# 3) exames_atrasados — encontre pacientes que disparam alerta
candidatos = conn.execute('SELECT paciente_id FROM pacientes ORDER BY paciente_id').fetchall()
encontradas = 0
for r in candidatos:
    alerts = tools.exames_atrasados(r['paciente_id'], conn)
    if alerts:
        encontradas += 1
        print(f'Paciente {r["paciente_id"]}:')
        for a in alerts:
            print(f'  - {a["tipo"]:<12} prioridade={a["prioridade"]:<5} {a["motivo"]}')
        if encontradas >= 5:
            break
print(f'\nTotal de pacientes com alertas (amostradas): {encontradas}+')

In [ ]:
# 4) consultar_medicamento
for termo in ['sertralina', 'contracepção', 'misoprostol', 'aborto']:
    print(f'\n>>> {termo}')
    for m in tools.consultar_medicamento(termo, conn):
        print(f'  {m["nome_principio_ativo"]:<35} cat. gestacão={m["categoria_gestacao"]} '
              f'lactacão={m["categoria_lactacao"]}')

In [ ]:
# 5) calendario_menstrual
for pid in [1, 2, 3, 10, 25]:
    r = tools.calendario_menstrual(pid, conn)
    print(f'Paciente {pid}: {r}')

In [ ]:
# 6) avaliar_padrao_violencia — três cenários
print('--- Sem alerta ---')
print(json.dumps(tools.avaliar_padrao_violencia(['somatizacoes_cronicas']), indent=2, ensure_ascii=False))

print('\n--- Atenção ---')
print(json.dumps(tools.avaliar_padrao_violencia([
    'somatizacoes_cronicas', 'baixa_adesao'
]), indent=2, ensure_ascii=False))

print('\n--- Alta suspeita ---')
print(json.dumps(tools.avaliar_padrao_violencia([
    'lesoes_inexplicadas', 'acompanhante_controlador',
    'discordancia_historia_exame', 'isolamento_social'
]), indent=2, ensure_ascii=False))

In [ ]:
# 7) consultar_violencia — verifica auditoria
tools.set_usuario_atual('dr_ana_residente')

# Sem motivo: rejeitado
print('Sem motivo:', tools.consultar_violencia(1, '', conn))

# Com motivo: registra acesso
pid_alvo = conn.execute('SELECT DISTINCT paciente_id FROM registros_violencia LIMIT 1').fetchone()['paciente_id']
print(f'\nConsultando paciente {pid_alvo} com motivo:')
for r in tools.consultar_violencia(pid_alvo, 'Atendimento ambulatorial — reavaliação', conn):
    print(' ', r)

# Log gerado
logs = conn.execute(
    'SELECT timestamp, usuario, tabela, paciente_id, motivo FROM log_acesso '
    'ORDER BY id DESC LIMIT 5'
).fetchall()
print('\nÚltimos logs de acesso:')
for l in logs:
    print(' ', dict(l))

In [ ]:
# 8) buscar_protocolo (RAG) — requer notebook 02 rodado
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True},
)
vs = Chroma(collection_name=COLLECTION, embedding_function=embeddings, persist_directory=CHROMA_DIR)
retriever = vs.as_retriever(search_kwargs={'k': 4})

for q in ['profilaxia HIV pós-violência sexual', 'critérios mamografia em mulher de 45 anos']:
    print(f'\n>>> {q}')
    for r in tools.buscar_protocolo(q, retriever):
        print(f'  [{r["category"]}] {r["doc_id"]}: {r["trecho"][:140]}...')

In [ ]:
# 9) Sanity check: as tools como StructuredTool do LangChain
lc_tools = tools.build_langchain_tools(conn, retriever)
for t in lc_tools:
    print(f'  {t.name:<26}  args={list(t.args.keys()) if t.args else "-"}')

# Chamada via interface LangChain (string args)
consult = next(t for t in lc_tools if t.name == 'consultar_prontuario')
print('\nVia LangChain tool:')
print(consult.invoke({'paciente_id': 1}))

In [ ]:
conn.close()
print('Testes concluídos.')